In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

## Load Data

In [2]:
data_aa = np.load('../change_aa.npy')
data_aa.shape

(5006, 19764)

In [3]:
# Data Loading
data_bb = np.load('../data_bb.npy')
torsion_angle = np.load('../torsion_angles.npy')[1:,:]
com_Evector = np.load('../com_e_vector.npy')
data_bb.shape, torsion_angle.shape, com_Evector.shape

((5006, 4941), (5006, 1647), (5006, 1647))

In [ ]:
def bin_data(values, num_bins=158, range_min=-3, range_max=3):
    bins = np.linspace(range_min, range_max, num_bins + 1)
    bin_indices = np.digitize(values, bins) - 1  # Subtract 1 to convert to 0-indexed
    return bin_indices

In [31]:

train_feature_bb = data_bb[:30,:].reshape(-1,3)
tor_train = torsion_angle[:30,:].reshape(-1,1)
com_train  = com_Evector[:30,:].reshape(-1,1)

train_feature_bb = np.concatenate((train_feature_bb,tor_train,com_train),axis=1)
train_feature_bb.shape

(49410, 5)

In [6]:
data_aa[0,:12]

array([ 0.757, -0.63 , -1.07 , -0.633, -0.47 , -0.63 , -0.713,  0.63 ,
        0.44 ,  0.137,  0.55 ,  1.3  ])

In [5]:
def get_labels(arr):
    start = -12
    for i in range(0,arr.shape[1],12):
        start = i
        end  = i+12
        print(start,end)
        if i ==0:
            label = arr[:,start:end]
        else:
            label = np.concatenate((label,arr[:,start:end]),axis=0)
    
    return label

In [6]:

train_label = get_labels(data_aa[:30,:])
train_label.shape

0 12
12 24
24 36
36 48
48 60
60 72
72 84
84 96
96 108
108 120
120 132
132 144
144 156
156 168
168 180
180 192
192 204
204 216
216 228
228 240
240 252
252 264
264 276
276 288
288 300
300 312
312 324
324 336
336 348
348 360
360 372
372 384
384 396
396 408
408 420
420 432
432 444
444 456
456 468
468 480
480 492
492 504
504 516
516 528
528 540
540 552
552 564
564 576
576 588
588 600
600 612
612 624
624 636
636 648
648 660
660 672
672 684
684 696
696 708
708 720
720 732
732 744
744 756
756 768
768 780
780 792
792 804
804 816
816 828
828 840
840 852
852 864
864 876
876 888
888 900
900 912
912 924
924 936
936 948
948 960
960 972
972 984
984 996
996 1008
1008 1020
1020 1032
1032 1044
1044 1056
1056 1068
1068 1080
1080 1092
1092 1104
1104 1116
1116 1128
1128 1140
1140 1152
1152 1164
1164 1176
1176 1188
1188 1200
1200 1212
1212 1224
1224 1236
1236 1248
1248 1260
1260 1272
1272 1284
1284 1296
1296 1308
1308 1320
1320 1332
1332 1344
1344 1356
1356 1368
1368 1380
1380 1392
1392 1404
1404 1416
1416 

(49410, 12)

In [88]:
# Determine the number of bins using Freedman-Diaconis rule
def freedman_diaconis_bins(y):
    q1 = np.percentile(y, 25)
    q3 = np.percentile(y, 75)
    iqr = q3 - q1
    bin_width = 2 * iqr / np.cbrt(len(y))
    num_bins = int(np.ceil((y.max() - y.min()) / bin_width))
    return num_bins

# Apply Freedman-Diaconis rule to bin target variable
def bin_data(y, num_bins):
    bin_edges = np.histogram_bin_edges(y, bins=num_bins)
    y_binned = np.digitize(y, bin_edges) - 1  # Subtract 1 to make bins zero-indexed
    y_binned = np.clip(y_binned, 0, num_bins - 1)  # Ensure indices are within valid range
    y_one_hot = to_categorical(y_binned, num_bins)
    return y_one_hot




In [7]:
# Function to bin data with specified bin width
def bin_data(values, bin_width=0.2, range_min=-3, range_max=3):
    num_bins = int((range_max - range_min) / bin_width)
    bins = np.linspace(range_min, range_max, num_bins + 1)
    bin_indices = np.digitize(values, bins) - 1  # Subtract 1 to convert to 0-indexed
    y_binned = np.clip(bin_indices, 0, num_bins - 1)  # Ensure indices are within valid range
    y_one_hot = to_categorical(y_binned, num_bins)
    return y_one_hot,num_bins

In [39]:
y_binned, numbin = bin_data(train_label,bin_width=0.3)
y_binned.shape,numbin

((49410, 12, 20), 20)

In [38]:
5*0.3

1.5

In [75]:
num_bins = freedman_diaconis_bins(train_label.reshape(-1,1))
num_bins

158

In [131]:
y_binned_list = []
num_bins_list = []
for i in range(1):
    
    y_binned = bin_data(train_label, 158)
    y_binned_list.append(y_binned)

In [98]:
[ 98,  48,  32,  48,  54,  48,  45,  93,  86,  76,  90, 117]

[98, 48, 32, 48, 54, 48, 45, 93, 86, 76, 90, 117]

In [132]:
np.array(y_binned_list).shape

(1, 823500, 12, 158)

In [133]:
y_binned_array = np.array(y_binned_list).reshape(823500, 12, 158)
y_binned_array.shape,  np.array(y_binned_list).shape

: 

: 

In [99]:
for i in y_binned_array[0,1]:
    print(i)

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
1.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [58]:
y_binned_array.shape

(49410, 12)

## DNN model

In [54]:
# Define the model architecture
def create_categorical_model(input_shape, num_bins):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=input_shape),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        # tf.keras.layers.Dense(num_bins, activation='softmax')  # Softmax activation for classification
        tf.keras.layers.Dense(num_bins, activation=None)  # logit
    ])
    return model

In [10]:
# Define the model architecture with logits for each output
def create_logit_model(input_shape, num_outputs):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Dense(64, activation='relu')(inputs)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    
    # Create separate output layers for each target
    outputs = []
    for i in range(1):
        output = tf.keras.layers.Dense(158, activation="softmax", name=f"output_{i}")(x)  # No activation for logits
        outputs.append(output)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

In [9]:
def create_logit_model(input_shape, num_outputs):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Dense(64, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    
    output = tf.keras.layers.Dense(num_outputs, activation="softmax", name="output")(x)
    
    
    model = tf.keras.Model(inputs=inputs, outputs=output)
    return model

In [40]:
# Example usage:
input_shape = (5,)  # Example input shape, adjust based on your actual data
output_num = 20  # Example number of bins, adjust based on your actual data

# Create the model
model = create_logit_model(input_shape, output_num)

In [41]:
model.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 5)]               0         
                                                                 
 dense_15 (Dense)            (None, 64)                384       
                                                                 
 batch_normalization_15 (Ba  (None, 64)                256       
 tchNormalization)                                               
                                                                 
 dense_16 (Dense)            (None, 128)               8320      
                                                                 
 batch_normalization_16 (Ba  (None, 128)               512       
 tchNormalization)                                               
                                                                 
 dropout_12 (Dropout)        (None, 128)               0   

In [74]:
num_bins," loss='categorical_crossentropy', metrics=['accuracy'])"

12

In [103]:
# Compile the model with sparse categorical crossentropy loss for each output
losses = {f'output_{i}': tf.keras.losses.CategoricalCrossentropy()
          for i in range(12)}
metrics = {f'output_{i}': 'accuracy' for i in range(12)}

model.compile(optimizer='adam', loss=losses, metrics=metrics)

In [19]:
import tensorflow as tf

def top_l_accuracy(y_true, y_pred, top_l=5):
    """
    Computes Top-L Accuracy metric.
    
    Parameters:
    - y_true: True labels (ground truth).
    - y_pred: Predicted probabilities or distances.
    - top_l: Number of top predictions to consider.
    
    Returns:
    - top_l_acc: Top-L Accuracy score.
    """
    # Ensure y_pred is a tensor
    y_pred = tf.convert_to_tensor(y_pred)
    
    # Extract true distances (or labels) and cast to integer type
    true_distances = tf.cast(y_true, tf.int32)
    
    # Get top-L predictions
    top_l_preds_indices = tf.math.top_k(y_pred, k=top_l).indices
    
    # Reshape true_distances to match top_l_preds_indices for comparison
    true_distances_expanded = tf.expand_dims(true_distances, axis=1)
    
    # Check if true distances are within top-L predicted distances
    true_in_top_l = tf.reduce_any(tf.equal(top_l_preds_indices, true_distances_expanded), axis=2)
    
    # Calculate Top-L Accuracy
    top_l_acc = tf.reduce_mean(tf.cast(true_in_top_l, tf.float32))
    
    return top_l_acc


In [42]:
def top_5_accuracy(y_true, y_pred):
  return tf.keras.metrics.top_k_categorical_accuracy(y_true, y_pred, k=5)

In [43]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', top_5_accuracy])

In [41]:
f1 = tf.keras.metrics.F1Score()

recall = tf.keras.metrics.Recall()
precision = tf.keras.metrics.Precision()

model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy', precision, recall, f1])

In [44]:
y_single = y_binned[:,0,:].reshape(-1,20)
y_single.shape, y_binned[:,0,:].shape

((49410, 20), (49410, 20))

In [22]:
tf.keras.metrics.top_k_categorical_accuracy

<function keras.src.metrics.accuracy_metrics.top_k_categorical_accuracy(y_true, y_pred, k=5)>

In [45]:
# Example training
history = model.fit(train_feature_bb,  # Input data
                   y_single,  # List of target arrays for each output
                    epochs=100,
                    batch_size=30) 



Epoch 1/100
1647/1647 [==============================] - 5s 2ms/step - loss: 2.7161 - accuracy: 0.0951 - top_5_accuracy: 0.4488
Epoch 2/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.5421 - accuracy: 0.1015 - top_5_accuracy: 0.4699
Epoch 3/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.5172 - accuracy: 0.1075 - top_5_accuracy: 0.4831
Epoch 4/100
1647/1647 [==============================] - 4s 3ms/step - loss: 2.5081 - accuracy: 0.1072 - top_5_accuracy: 0.4854
Epoch 5/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.4975 - accuracy: 0.1148 - top_5_accuracy: 0.4936
Epoch 6/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.4899 - accuracy: 0.1180 - top_5_accuracy: 0.4992
Epoch 7/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.4781 - accuracy: 0.1247 - top_5_accuracy: 0.5132
Epoch 8/100
1647/1647 [==============================] - 4s 2ms/step - loss: 2.4727 - accuracy: 0.1287 -

In [85]:
y_binned_array[0]

array([ 98,  48,  32,  48,  54,  48,  45,  93,  86,  76,  90, 117])

In [1]:
# Example training
history = model.fit(train_feature_bb,  # Input data
                    [y_binned[:, i,:] for i in range(12)],  # List of target arrays for each output
                    epochs=100,
                    batch_size=8) 


NameError: name 'model' is not defined